# Dimension accounts


In [0]:
#Imports
from pyspark.sql.functions import col
from delta.tables import DeltaTable

In [0]:
accounts_df = spark.read.table("neo_bank.silver.accounts")
branches_df = spark.read.table("neo_bank.gold.dim_branches")
customers_df = spark.read.table("neo_bank.gold.dim_customers")


In [0]:
dim_accounts_df = (
    accounts_df.alias("a")
        .join(
            branches_df.alias("b"),
            col("a.branch_code")==col("b.branch_code"),
            "left"
        )
        .join(
            customers_df.alias("c"),
            col("a.customer_id")==col("c.customer_id"),
            "left"
        )
        .select(
            col("a.account_id"),
            col("c.customer_sk"),
            col("a.account_type"),
            col("a.balance"),
            col("a.currency"),
            col("b.branch_sk").alias("account_branch_sk"),
            col("a.status"),
            col("opened_date").alias("account_opened_date")
        )
)

In [0]:
dim_table = DeltaTable.forName(spark, "neo_bank.gold.dim_accounts")

(
    dim_table.alias("target")
    .merge(
        dim_accounts_df.alias("source"),
        condition="target.account_id = source.account_id"
    )
    .whenMatchedUpdate(set={
        "customer_sk":"source.customer_sk",
        "account_type":"source.account_type",
        "balance":"source.balance",
        "currency":"source.currency",
        "account_branch_sk":"source.account_branch_sk",
        "status":"source.status",
        "account_opened_date":"source.account_opened_date"
    })
    .whenNotMatchedInsert(values={
        "account_id":"source.account_id",
        "customer_sk":"source.customer_sk",
        "account_type":"source.account_type",
        "balance":"source.balance",
        "currency":"source.currency",
        "account_branch_sk":"source.account_branch_sk",
        "status":"source.status",
        "account_opened_date":"source.account_opened_date"
    })
    .execute()
)